In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [2]:
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
tlc_taxi_zone_lookup = pd.read_csv("../data/mappings/taxi_zone_lookup.csv")
print(tlc_taxi_zone_lookup.shape[0], "rows,", tlc_taxi_zone_lookup.shape[1], "columns")
tlc_taxi_zone_lookup.head()

265 rows, 4 columns


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
FILE_PATH = r"../data/raw/yellow_tripdata_2026-05.parquet"
df = pd.read_parquet(FILE_PATH)
print(f"{df.shape[0]} rows, {df.shape[1]} columns")

4090836 rows, 20 columns


In [5]:
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [6]:
df = df[["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "PULocationID", "DOLocationID", "payment_type", "fare_amount", "extra", "mta_tax","tip_amount", "tolls_amount", "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]]

df = df.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id"
})

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

payment_type_mapping = {
    0: "Flex Fare trip",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}
df['payment_type'] = df['payment_type'].map(payment_type_mapping)

In [7]:
df.isna().sum()

pickup_datetime               0
dropoff_datetime              0
passenger_count          955371
trip_distance                 0
pickup_location_id            0
dropoff_location_id           0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     955371
Airport_fee              955371
cbd_congestion_fee            0
dtype: int64

In [8]:
# --- Pickup and Dropoff Datetime Columns ---
print(df['pickup_datetime'].min(), df['pickup_datetime'].max())
print(df['dropoff_datetime'].min(), df['dropoff_datetime'].max())

2008-12-31 23:05:53 2026-06-01 00:20:35
2009-01-01 17:08:07 2026-06-02 12:53:42


In [9]:
# --- Passenger Count Column ---
print(df['passenger_count'].value_counts())
print(df['passenger_count'].value_counts().sum())

passenger_count
1.00    2578718
2.00     395103
3.00      83959
4.00      52929
0.00      12533
5.00       7666
6.00       4553
8.00          3
9.00          1
Name: count, dtype: int64
3135465


In [10]:
# --- Trip Distance Column ---
print(df['trip_distance'].value_counts())
print(df['trip_distance'].value_counts().sum())

trip_distance
0.00         113031
0.90          40207
1.00          39612
0.80          39304
1.10          38820
1.20          37207
0.70          37061
1.30          35785
1.40          33603
1.50          32373
0.60          32335
1.60          30890
1.70          28948
1.80          26916
0.50          25852
1.90          25113
2.00          23306
2.10          22124
2.20          20347
2.30          18754
0.40          18381
2.40          17598
2.50          16145
2.60          15191
0.01          15071
2.70          14185
2.80          13295
2.90          12313
0.93          12092
0.92          12072
0.86          12021
0.96          12015
0.87          11969
1.01          11969
0.91          11937
0.97          11918
0.98          11918
1.04          11917
0.84          11899
1.02          11886
0.85          11851
0.94          11851
0.88          11843
1.06          11838
0.95          11837
1.03          11829
0.99          11822
0.89          11787
0.81          11781
1.07  

In [11]:
# --- Pickup and Dropoff Location id Columns ---
print(df['pickup_location_id'].value_counts())
print(df['dropoff_location_id'].value_counts())

pickup_location_id
237    198008
236    175969
161    161315
132    149312
142    124523
162    120488
186    113921
230    111934
239    108689
79     107374
138    106388
170    102755
234    101697
163    100524
68      99835
141     91725
249     89571
48      89093
238     80807
140     78479
263     77369
164     76704
107     74684
246     74631
229     64770
90      60980
262     59916
114     59066
148     56529
231     56346
113     55962
43      54474
100     49800
143     48875
137     47174
144     44972
158     42773
233     42050
151     35020
75      34338
211     33427
166     30217
87      29127
50      28135
13      24652
41      21083
125     19814
74      18016
4       17736
261     17472
42      15194
88      15040
232     14286
24      13801
244     10841
224     10796
255     10447
45      10285
116      9673
145      9042
209      8894
70       8621
7        7776
61       7527
226      7010
256      6932
37       6675
112      6587
152      6565
97       6216
3

In [12]:
# --- Payment Type Column ---
print(df['payment_type'].value_counts())
print(df['payment_type'].value_counts().sum())

payment_type
Credit card       2727585
Flex Fare trip     955371
Cash               372909
Dispute             22987
No charge           11984
Name: count, dtype: int64
4090836


In [13]:
print(df[['fare_amount', 'tip_amount', 'total_amount']].describe())

       fare_amount  tip_amount  total_amount
count   4090836.00  4090836.00    4090836.00
mean         21.51        2.98         30.49
std          19.01        4.02         22.97
min        -950.00      -47.10       -951.00
25%          10.00        0.00         17.64
50%          16.30        2.24         23.94
75%          26.80        4.10         34.95
max        5525.99      239.00       5530.74


### Empty Passenger Counts

In [14]:
empty_passenger_count = df[df['passenger_count'].isna()]
empty_passenger_count.shape[0]

955371

In [15]:
empty_passenger_count["payment_type"].value_counts()

payment_type
Flex Fare trip    955371
Name: count, dtype: int64

In [16]:
empty_passenger_count[["fare_amount", "tip_amount", "total_amount"]].describe()

,fare_amount,tip_amount,total_amount
count,955371.00,955371.00,955371.00
mean,25.83,0.48,32.74
std,14.33,1.91,16.99
min,-18.30,0.00,-23.05
25%,16.19,0.00,21.74
50%,23.20,0.00,29.11
75%,32.21,0.00,39.56
max,1061.72,67.20,1065.72


### Zero Passenger Count

In [17]:
zero_passenger_count = df[df['passenger_count']==0]
zero_passenger_count.shape[0]

12533

In [18]:
zero_passenger_count["trip_distance"].value_counts()

trip_distance
0.90      562
1.00      514
0.70      514
0.80      477
1.10      475
0.00      449
1.20      446
0.60      438
1.30      419
1.40      388
0.50      378
1.50      375
1.60      356
1.70      343
1.80      329
0.40      280
1.90      269
2.00      258
2.10      247
2.20      196
2.30      195
2.40      181
0.30      162
2.60      159
2.50      152
2.70      140
2.80      132
3.10      122
3.20      101
3.00      101
2.90       97
3.30       90
3.40       87
0.20       76
3.50       71
3.70       69
3.60       56
3.90       56
3.80       45
4.10       44
4.00       44
4.20       43
4.50       41
0.10       40
4.30       34
4.80       33
4.60       32
4.40       32
4.70       31
4.90       28
5.10       26
5.00       25
5.30       24
5.40       24
5.50       22
5.20       21
5.60       21
6.20       20
8.20       20
6.30       19
9.70       19
5.90       19
7.30       18
6.00       18
6.70       18
6.10       18
7.50       16
5.70       16
8.40       15
11.70      15
7.20  

In [19]:
zero_passenger_count[['fare_amount', 'tip_amount', 'total_amount']].describe()

,fare_amount,tip_amount,total_amount
count,12533.00,12533.00,12533.00
mean,17.39,3.31,26.30
std,20.53,3.75,23.65
min,-32.10,-0.90,-34.35
25%,8.60,1.00,15.55
50%,12.80,2.80,20.55
75%,19.80,4.29,28.85
max,1400.00,150.00,1400.00


### Zero trip distance

In [20]:
zero_trip_distance = df[df['trip_distance']==0]
zero_trip_distance.shape[0]

113031

In [21]:
zero_trip_distance["passenger_count"].value_counts()

passenger_count
1.00    28491
2.00     4539
4.00     2412
3.00     1104
0.00      449
5.00       46
6.00       21
8.00        2
9.00        1
Name: count, dtype: int64

In [22]:
zero_trip_distance[['fare_amount', 'tip_amount', 'total_amount']].describe()

,fare_amount,tip_amount,total_amount
count,113031.00,113031.00,113031.00
mean,29.75,1.61,36.40
std,36.40,4.95,38.69
min,-950.00,-14.95,-951.00
25%,14.00,0.00,19.23
50%,23.55,0.00,29.25
75%,37.12,0.00,43.93
max,5525.99,239.00,5530.74


### Negative Fares

In [23]:
negative_fares = df[(df['fare_amount'] < 0) | (df['tip_amount'] < 0) | (df['total_amount'] < 0)]
negative_fares.shape[0]

14888

In [24]:
negative_fares["payment_type"].value_counts()

payment_type
Dispute           9044
Cash              3850
No charge         1620
Credit card        361
Flex Fare trip      13
Name: count, dtype: int64

In [25]:
negative_fares[["trip_distance","fare_amount","tip_amount","total_amount"]].describe()

,trip_distance,fare_amount,tip_amount,total_amount
count,14888.00,14888.00,14888.00,14888.00
mean,3.33,-22.15,-0.02,-28.07
std,5.73,32.56,0.72,34.18
min,0.00,-950.00,-47.10,-951.00
25%,0.25,-25.40,0.00,-31.25
50%,1.13,-11.40,0.00,-17.15
75%,3.18,-5.10,0.00,-10.50
max,93.18,0.00,4.51,40.76


### Pickup & Dropoff Datetime

In [26]:
df[df['pickup_datetime'] < datetime(2026,5,1)]

,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
40,2026-04-30 23:53:41,2026-05-01 00:14:20,2.00,8.76,138,161,Credit card,35.90,6.00,0.50,11.22,7.46,1.00,67.33,2.50,2.00,0.75
84,2026-04-30 23:54:52,2026-05-01 00:11:11,5.00,2.93,239,166,Credit card,18.40,1.00,0.50,4.68,0.00,1.00,28.08,2.50,0.00,0.00
163,2026-04-30 23:59:17,2026-05-01 00:12:36,1.00,2.42,144,170,Credit card,14.90,1.00,0.50,4.13,0.00,1.00,24.78,2.50,0.00,0.75
390,2026-04-30 23:59:17,2026-05-01 00:27:55,1.00,5.69,209,236,Cash,29.60,1.00,0.50,0.00,0.00,1.00,35.35,2.50,0.00,0.75
585,2026-04-30 23:51:13,2026-05-01 00:28:20,5.00,9.58,88,41,Credit card,47.80,1.00,0.50,0.00,0.00,1.00,53.55,2.50,0.00,0.75
946,2026-04-30 23:51:53,2026-04-30 23:58:30,1.00,1.06,170,137,Credit card,8.60,1.00,0.50,0.05,0.00,1.00,14.40,2.50,0.00,0.75
958,2026-04-30 23:47:18,2026-04-30 23:53:05,5.00,1.24,50,246,Credit card,7.90,1.00,0.50,4.10,0.00,1.00,17.75,2.50,0.00,0.75
1310,2026-04-30 23:49:52,2026-05-01 00:04:47,1.00,1.53,68,234,Cash,13.50,1.00,0.50,0.00,0.00,1.00,19.25,2.50,0.00,0.75
1402,2026-04-30 23:54:59,2026-05-01 00:12:16,1.00,2.27,114,87,Cash,13.50,1.00,0.50,0.00,0.00,1.00,19.25,2.50,0.00,0.75
2125,2026-04-30 23:54:59,2026-05-01 00:36:14,1.00,16.61,138,14,Cash,68.10,6.00,0.50,0.00,0.00,1.00,77.60,0.00,2.00,0.00


### Clean Suspicious Data

In [27]:
print(f"Current DataFrame shape: {df.shape}")
cleaned_df = df[ (df["fare_amount"] >= 0) & (df["tip_amount"] >= 0) & (df["total_amount"] >= 0)].copy()
print(f"DataFrame shape after dropping negative fares: {cleaned_df.shape}")
cleaned_df = cleaned_df[cleaned_df["passenger_count"].isna() | (cleaned_df["passenger_count"] > 0)]
print(f"DataFrame shape after dropping 0 passengers: {cleaned_df.shape}")
cleaned_df = cleaned_df[cleaned_df["trip_distance"]>0]
print(f"DataFrame shape after dropping 0 trip distances: {cleaned_df.shape}")
cleaned_df = cleaned_df[cleaned_df['pickup_datetime'] >= datetime(2026,5,1)]
print(f"DataFrame shape after dropping pickup dates older than 2026-05-01: {cleaned_df.shape}")

Current DataFrame shape: (4090836, 17)
DataFrame shape after dropping negative fares: (4075948, 17)
DataFrame shape after dropping 0 passengers: (4063425, 17)
DataFrame shape after dropping 0 trip distances: (3952609, 17)
DataFrame shape after dropping pickup dates older than 2026-05-01: (3952596, 17)


In [28]:
cleaned_df.isna().sum()

pickup_datetime               0
dropoff_datetime              0
passenger_count          879392
trip_distance                 0
pickup_location_id            0
dropoff_location_id           0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     879392
Airport_fee              879392
cbd_congestion_fee            0
dtype: int64

In [29]:
cleaned_df["passenger_count"].value_counts()

passenger_count
1.00    2540002
2.00     388529
3.00      82398
4.00      50155
5.00       7599
6.00       4520
8.00          1
Name: count, dtype: int64

In [30]:
cleaned_df["payment_type"].value_counts()

payment_type
Credit card       2695650
Flex Fare trip     879392
Cash               358216
Dispute             11931
No charge            7407
Name: count, dtype: int64

In [31]:
cleaned_df[["pickup_datetime","dropoff_datetime","trip_distance","pickup_location_id","dropoff_location_id","fare_amount","tip_amount","total_amount"]].describe()

,pickup_datetime,dropoff_datetime,trip_distance,pickup_location_id,dropoff_location_id,fare_amount,tip_amount,total_amount
count,3952596,3952596,3952596.00,3952596.00,3952596.00,3952596.00,3952596.00,3952596.00
mean,2026-05-16 10:21:31.977615,2026-05-16 10:40:25.085140,5.11,162.23,161.95,21.44,3.02,30.53
min,2026-05-01 00:00:00,2026-05-01 00:03:33,0.01,1.00,1.00,0.00,0.00,0.00
25%,2026-05-08 21:14:23,2026-05-08 21:31:39,1.10,114.00,112.00,10.00,0.00,17.65
50%,2026-05-16 01:46:50,2026-05-16 02:02:21.500000,1.93,161.00,162.00,16.20,2.33,23.86
75%,2026-05-23 15:46:11,2026-05-23 16:04:07,3.93,234.00,234.00,26.32,4.15,34.81
max,2026-06-01 00:20:35,2026-06-02 12:53:42,307491.47,265.00,265.00,5525.99,222.00,5530.74
std,NaN,NaN,492.00,66.88,70.70,18.01,3.98,22.05


### Mismatch Totals

In [94]:
def calculate_total_amount(row):
    if row["payment_type"] in ["Dispute", "No charge"]:
        return row["fare_amount"] + row["tip_amount"] + row["extra"] + row["mta_tax"] + row["tolls_amount"] + row["improvement_surcharge"] + row["congestion_surcharge"] + row["Airport_fee"] + row["cbd_congestion_fee"]
    elif row["payment_type"] in ["Cash"]:
            return row["fare_amount"] + row["extra"] + row["mta_tax"] + row["tolls_amount"] + row["improvement_surcharge"]
    
    return row["fare_amount"] + row["tip_amount"] + row["extra"] + row["mta_tax"] + row["tolls_amount"] + row["improvement_surcharge"]

cleaned_df["calc_total_amount"] = cleaned_df.apply(calculate_total_amount, axis=1)

cleaned_df["total_amount_check"] = cleaned_df["calc_total_amount"] == cleaned_df["total_amount"]

cleaned_df["total_amount_check"].value_counts()



total_amount_check
False    3195344
True      757252
Name: count, dtype: int64

In [62]:
cleaned_df["difference"] = (
    cleaned_df["total_amount"] - cleaned_df["calc_total_amount"]
)
cleaned_df["difference"] = cleaned_df["difference"].round(2)
cleaned_df["difference"].describe()



count   3952596.00
mean          2.90
std           3.60
min         -55.74
25%           2.50
50%           3.25
75%           3.25
max         323.89
Name: difference, dtype: float64

In [74]:
filtered_df = cleaned_df[(cleaned_df['difference'] != 0.00) & (cleaned_df['difference'].notna())][["passenger_count","trip_distance", "payment_type","fare_amount", "tip_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee", "total_amount", "calc_total_amount", "difference"]]
filtered_df.shape

(3143560, 15)

In [75]:
filtered_df["possible_difference_charges_incl_improvement_surcharge"] = filtered_df["improvement_surcharge"] + filtered_df["congestion_surcharge"] + filtered_df["Airport_fee"] + filtered_df["cbd_congestion_fee"]

filtered_df["possible_difference_charges_excl_improvement_surcharge"] = filtered_df["congestion_surcharge"] + filtered_df["Airport_fee"] + filtered_df["cbd_congestion_fee"]

In [76]:
filtered_df["payment_type"].value_counts()

payment_type
Credit card       2061885
Flex Fare trip     789116
Cash               283881
No charge            5478
Dispute              3200
Name: count, dtype: int64

In [84]:
filtered_df[abs(filtered_df['difference'])==filtered_df['possible_difference_charges_incl_improvement_surcharge']].shape[0]

11210

In [85]:
filtered_df[abs(filtered_df['difference'])==filtered_df['possible_difference_charges_excl_improvement_surcharge']].shape[0]

2330120

In [88]:
filtered_df[abs(filtered_df['difference'])==filtered_df['improvement_surcharge']].shape[0]

2638

In [90]:
filtered_df[
    (abs(filtered_df['difference'])!=filtered_df['possible_difference_charges_excl_improvement_surcharge']) & 
    (abs(filtered_df['difference'])!=filtered_df['possible_difference_charges_incl_improvement_surcharge']) & 
    (abs(filtered_df['difference'])!=filtered_df['improvement_surcharge'])
    ].shape[0]

799985

In [92]:
cleaned_df[cleaned_df['payment_type']=='Cash']['tip_amount'].value_counts()

tip_amount
0.00     358191
4.91          2
5.68          1
9.81          1
8.07          1
20.20         1
2.31          1
6.59          1
2.81          1
6.40          1
3.15          1
5.55          1
4.00          1
2.67          1
8.40          1
6.87          1
2.50          1
2.73          1
2.25          1
6.31          1
4.63          1
2.00          1
3.85          1
1.00          1
7.15          1
Name: count, dtype: int64

In [ ]:
cleaned_df = cleaned_df.drop(columns=["calc_total_amount", "total_amount_check", "difference"])
cleaned_df.columns

Index(['pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'pickup_location_id', 'dropoff_location_id',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee'],
      dtype='str')

In [97]:
cleaned_df.to_parquet("../data/cleaned/NYC_Trips.parquet")

In [4]:
import pandas as pd
temp = pd.read_parquet("../data/cleaned/NYC_Trips.parquet")
temp = temp.rename(columns={"Airport_fee":"airport_fee"})
temp.to_parquet("../data/cleaned/NYC_Trips.parquet")